# Michaelis-Menten — Tau Hybrid
***
## Setup the Environment
***

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from copy import deepcopy
sys.path.insert(1, os.path.abspath(os.path.join(os.getcwd(), '../')))

In [3]:
import gillespy3d_pp as gillespy2

In [4]:
gillespy2.__file__

'/Users/anisgolriz/Desktop/GillesPy3D/simulation_lib/gillespy3d_pp/__init__.py'

In [5]:
def create_michaelis_menten(sub0=301, enz0=120, rate1=0.0017, rate2=0.5, rate3=0.1, end_t=100):
    # Initialize Model
    model = gillespy2.Model(name="Michaelis_Menten")

    # Define Variables (GillesPy2.Species)
    A = gillespy2.Species(name='Substrate', initial_value=sub0)
    B = gillespy2.Species(name='Enzyme', initial_value=enz0)
    C = gillespy2.Species(name='Enzyme_Substrate_Complex', initial_value=0)
    D = gillespy2.Species(name='Product', initial_value=0)

    # Add Variables to Model
    model.add_species([A, B, C, D])

    # Define Parameters
    p1 = gillespy2.Parameter(name='rate1', expression=rate1)
    p2 = gillespy2.Parameter(name='rate2', expression=rate2)
    p3 = gillespy2.Parameter(name='rate3', expression=rate3)

    # Add Parameters to Model
    model.add_parameter([p1, p2, p3])

    # Define Reactions
    r1 = gillespy2.Reaction(
        name="r1", reactants={'Substrate': 1, 'Enzyme': 1},
        products={'Enzyme_Substrate_Complex': 1}, rate='rate1'
    )
    r2 = gillespy2.Reaction(
        name="r2", reactants={'Enzyme_Substrate_Complex': 1},
        products={'Substrate': 1, 'Enzyme': 1}, rate='rate2'
    )
    r3 = gillespy2.Reaction(
        name="r3", reactants={'Enzyme_Substrate_Complex': 1},
        products={'Enzyme': 1, 'Product': 1}, rate='rate3'
    )
    # Add Reactions to Model
    model.add_reaction([r1, r2, r3])

    # Define Timespan
    tspan = gillespy2.TimeSpan.linspace(end_t=end_t, num_points=int(end_t) + 1)

    # Set Model Timespan
    model.timespan(tspan)
    return model

### Instantiate the Model

### Parameters with Avogadro

In [7]:
# --- WITH Avogadro's number ---
nA  = 6.023e23     # Avogadro's number
vol = 1e-15        # system volume (L)

sub_conc, enz_conc = 5e-7, 2e-7     # molar (paper concentrations)
k1, k2, k3 = 1e6, .5, 0.1         # deterministic rate constants

sub0  = round(sub_conc * nA * vol)  # 301
enz0  = round(enz_conc * nA * vol)  # 120
rate1 = k1 / (nA * vol)             # ~0.00166
rate2 = k2
rate3 = k3
end_t = 100

model = create_michaelis_menten(sub0, enz0, rate1, rate2, rate3, end_t)

### Parameters without Avogadro

In [ ]:
# --- WITHOUT Avogadro's number ---
sub0, enz0 = 301, 120
rate1, rate2, rate3 = 0.0017, 0.5, 0.1
end_t = 100

model = create_michaelis_menten(sub0, enz0, rate1, rate2, rate3, end_t)

In [8]:
dir(gillespy2)

['Ensemble',
 'Model',
 'NumPySSASolver',
 'Parameter',
 'Reaction',
 'Result',
 'Simulation',
 'Species',
 'TimeSpan',
 '__author__',
 '__builtins__',
 '__cached__',
 '__copyright__',
 '__description__',
 '__doc__',
 '__email__',
 '__file__',
 '__license__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__title__',
 '__url__',
 '__version__',
 'assignmentrule',
 'build_ensemble',
 'core',
 'error',
 'functiondefinition',
 'log',
 'logging',
 'model',
 'ode_solver',
 'parameter',
 'raterule',
 'reaction',
 'result',
 'simulation',
 'solvers',
 'sortableobject',
 'species',
 'sys',
 'tau_hybrid_solver',
 'tau_leaping_solver',
 'timespan',
 'utils',
 'version']

In [9]:
import matplotlib.pyplot as plt

sim = gillespy2.Simulation(model, number_of_trajectories=10, dt=.1, end_t=end_t, solver="HYBRID")

num_traj = 10
traj_data = []
for traj in range(num_traj):
    sim.reset()  # reset the simulation after each run
    times = []
    substrate = []
    enzyme = []
    complex_ = []
    product = []
    prev_t = -1
    while sim.get_time() < sim.end_t:
        t = sim.get_time()
        if t == prev_t:
            raise RuntimeError("Time is not advancing")
        prev_t = t
        times.append(sim.get_time())
        substrate.append(sim.get_species('Substrate'))
        enzyme.append(sim.get_species('Enzyme'))
        complex_.append(sim.get_species('Enzyme_Substrate_Complex'))
        product.append(sim.get_species('Product'))
        sim.run_until(sim.get_time() + sim.dt)
    traj_data.append([times, substrate, enzyme, product, complex_])

plt.figure(figsize=(8, 5))

for i, (times, substrate, enzyme, product, complex_) in enumerate(traj_data):
    if i == 0:
        plt.plot(times, substrate, label='Substrate', color='red', alpha=.9)
        plt.plot(times, enzyme, label='Enzyme', color='blue', alpha=.9)
        plt.plot(times, complex_, label='ES Complex', color='green', alpha=.9)
        plt.plot(times, product, label='Product', color='black', alpha=.9)
    else:
        plt.plot(times, substrate, label='', color='red', alpha=.2)
        plt.plot(times, enzyme, label='', color='blue', alpha=.2)
        plt.plot(times, complex_, label='', color='green', alpha=.2)
        plt.plot(times, product, label='', color='black', alpha=.2)

plt.xlabel('Time')
plt.ylabel('Molecule Count')
plt.title('Michaelis–Menten Tau Hybrid Simulation')
plt.legend()
plt.grid(True)
plt.show()

AttributeError: 'TauHybridSolver' object has no attribute 'run_until'

In [ ]:
# Step-through printout
sim = gillespy2.Simulation(model, number_of_trajectories=10, dt=.1, end_t=10, solver="HYBRID")

for traj in range(num_traj):
    sim.reset()  # reset the simulation after each run
    while sim.get_time() < sim.end_t:
        print('traj', traj, ' t:', sim.get_time(), ' Substrate:', sim.get_species('Substrate'))
        sim.run_until(sim.get_time() + sim.dt)

In [ ]:
plt.plot(times)

In [ ]:
substrate[-1]